# ARC-AGI-2 as block objects

This notebook demonstrates that FeatureGraph's object construction carries from
one-dimensional observation sequences to two-dimensional discrete grids, and shows
what the representation reports when it reaches the edge of what it can describe.

It is a **representation demonstration, not a solver**. The question is not "was
the task solved" but "where does a declared operator vocabulary apply, and where
does it run out".

The construction is the same four stages the temporal behaviors use:

```text
grid cell observations
    → per-cell operator match states
    → block boundaries and identities
    → one row per block object
    → computational queries
```

In [ ]:
import pandas as pd

import featuregraph as fg
from featuregraph.behaviors.composition import BlockComposition, resolve_layout

pd.set_option("display.width", 160)

GROUP = ["task_id", "pair_type", "pair_index"]


def block_objects(task_id, split="training", background_color=0):
    """Construct block objects for one ARC-AGI-2 task."""
    observations = fg.datasets.arc_agi(task_id, split=split)
    builder = BlockComposition(
        signals="color",
        group=GROUP,
        background_color=background_color,
    )
    return builder, builder.summarize(builder.fit_transform(observations))

## 1. Observations

The dataset loader returns one row per grid cell. Nothing is detected yet — this
is what the data contains.

In [ ]:
observations = fg.datasets.arc_agi("00576224", split="training")
observations.head()

In [ ]:
observations.groupby(["pair_type", "pair_index", "grid_role"])[
    ["grid_height", "grid_width"]
].first()

## 2. Block objects

`BlockComposition` aligns each output cell to the input cell it could have come
from, evaluates every declared operator against it, and lifts those per-cell
states to whole blocks. The result is one row per block.

In [ ]:
builder, objects = block_objects("00576224")

objects.to_pandas()[
    [
        "pair_type",
        "pair_index",
        "block_row",
        "block_column",
        "cell_count",
        "candidate_count",
        "operator",
    ]
].head(12)

### Candidate sets are retained, not collapsed

A block is described by *every* operator that reproduces it exactly. When an input
grid happens to be symmetric, several operators are indistinguishable on it. That
ambiguity is a property of the observation, so it is recorded rather than resolved
away or raised as an error.

In [ ]:
table = objects.to_pandas()
ambiguous = table[table["candidate_count"] > 1]

ambiguous[
    ["pair_type", "pair_index", "block_row", "block_column", "candidates"]
]

### Blocks the vocabulary cannot describe are queryable

A block matched by no declared operator is an object with an empty candidate set.
Finding those is a query over the object table, not a search through exception
messages — and it reports *every* such block, not just the first one.

In [ ]:
_, unmatched_example = block_objects("0692e18c")

unmatched = (
    unmatched_example.query()
    .where(pair_type="train", is_unmatched=True)
    .select("pair_index", "block_row", "block_column", "cell_count")
    .collect()
)

print(f"{len(unmatched)} of {unmatched_example.count} blocks are undescribed")
unmatched.head()

## 3. Resolving a layout is a choice of grouping

Intersecting candidate sets over the demonstration pairs yields an instruction
layout. **The grouping selects the hypothesis class.**

Layouts are resolved from demonstrations only, so a resolved layout never sees its
own answer.

In [ ]:
demonstrations = objects.query().where(pair_type="train").collect()

resolve_layout(demonstrations, by=("block_row", "block_column"))[
    ["block_row", "block_column", "candidate_count", "operator"]
]

Grouping by block position asks for a layout fixed to position, and here it
resolves: three operators ambiguous on one pair are pinned down by another.

### When a fixed layout is the wrong question

Task `007bbfb7` tiles a copy of the grid wherever the input cell is non-background.
Its layout therefore *changes with the input*. Grouping by block position asks a
question the task does not answer, and the intersection empties out.

In [ ]:
_, fractal = block_objects("007bbfb7")
fractal_demonstrations = fractal.query().where(pair_type="train").collect()

by_position = resolve_layout(
    fractal_demonstrations,
    by=("block_row", "block_column"),
)
by_position[["block_row", "block_column", "candidate_count", "operator"]]

Every block coordinate resolves to the empty set. Under a solver this is a
`ValueError` and the task is lost.

The same objects carry the state of the input cell each block corresponds to.
Grouping on that instead asks the question the task *does* answer:

In [ ]:
by_state = resolve_layout(fractal_demonstrations, by=("block_state",))
by_state[["block_state", "candidate_count", "operator"]]

Background cells produce background blocks; foreground cells produce copies.

These are the two "solver families" in `featuregraph.utils._arc_agi` — but they are
not two algorithms. They are two `GROUP BY` clauses over one object table. A third
hypothesis would be a third grouping, not a third solver.

## 4. Pairs that fall outside the frame

A pair whose output does not tile its input cannot be described as block
composition at all. That is recorded with a reason rather than raised, so a task
can be partly described and still reported.

In [ ]:
out_of_frame_builder, out_of_frame = block_objects("017c7c7b")
out_of_frame_builder.declined_

### Test pairs are described, but never used to resolve a layout

The public ARC-AGI-2 files include test outputs, so test pairs *do* produce block
objects here. That makes the guard explicit rather than incidental: every layout
above is resolved from `pair_type == "train"` only. Nothing stops you from
intersecting the test pair in as well — which is exactly why the filter is written
out each time.

A pair that genuinely carries no output grid is declined as `no_output_grid`, a
different situation from a pair the representation cannot describe.

In [ ]:
objects.to_pandas().groupby("pair_type").size().rename("block_objects")

## 5. The result across the whole benchmark

`experiments/arc/describability.py` runs this construction over all 1,120 public
ARC-AGI-2 tasks. The recorded artifact:

In [ ]:
describability = pd.read_csv("../artifacts/arc/describability.csv")

summary = []
for split, part in describability.groupby("split"):
    in_frame = part[part["in_frame"]]
    non_trivial = in_frame[~in_frame["identity_layout"].astype(bool)]
    summary.append(
        {
            "split": split,
            "tasks": len(part),
            "in_frame": len(in_frame),
            "identity_layout": len(in_frame) - len(non_trivial),
            "non_trivial_tiling": len(non_trivial),
        }
    )

pd.DataFrame(summary)

**The block-composition family is absent from the evaluation split.** Not rarer,
not harder — absent. All 81 evaluation tasks that nominally sit in frame are
same-shape, where a 1×1 block layout asserts nothing.

A solver benchmark would have reported this as a low score. The describability
table reports it as what it is: the structure taught by the training split does
not occur in the evaluation split in the form it was learned.

In [ ]:
same_shape = describability[
    describability["in_frame"] & describability["identity_layout"].astype(bool)
]

(
    same_shape.groupby("split").size()
    / describability.groupby("split").size()
).rename("share_same_shape")

Roughly two thirds of *both* splits are same-shape tasks. Block decomposition is
the wrong object boundary for them — their objects are connected regions, not
blocks. That is the next construction to build, and it is this same pipeline with
a different identity rule.

See [`artifacts/arc/README.md`](../artifacts/arc/README.md) for the full study
record, including limits.